# Knowledge Tracing Evaluation
**BKT (Bayesian Knowledge Tracing) vs Elo Baseline**


In [ ]:
import sys, json
from pathlib import Path
import os
sys.path.insert(0, os.getcwd())

from tutor.curriculum_loader import CurriculumLoader
from tutor.adaptive import (
    EloTracker, simulate_learner_replay, roc_auc, evaluate
)

SEED_PATH = "T3.1_Math_Tutor/curriculum_seed.json"
loader = CurriculumLoader(SEED_PATH)
print(f"Curriculum: {loader.summary()}")

[CurriculumLoader] Loaded 73 items across skills: {'counting', 'number_sense', 'addition', 'word_problem', 'subtraction'}
Curriculum: {'total': 73, 'by_skill': {'counting': 10, 'number_sense': 9, 'addition': 30, 'subtraction': 16, 'word_problem': 8}, 'by_difficulty': {1: 9, 2: 14, 3: 20, 5: 4, 4: 11, 6: 4, 8: 4, 7: 2, 9: 3, 10: 2}}


## Single Learner Simulation

In [5]:
records = simulate_learner_replay(loader.items, n_steps=40, seed=0)
print(f"Simulated {len(records)} response events")
print(f"Sample records:")
for r in records[:5]:
    print(f"  skill={r['skill']:12s} diff={r['difficulty']}  "
          f"correct={r['correct']}  bkt_pred={r['bkt_pred']:.3f}  elo_pred={r['elo_pred']:.3f}")

Simulated 40 response events
Sample records:
  skill=word_problem diff=5  correct=0  bkt_pred=0.220  elo_pred=0.429
  skill=word_problem diff=6  correct=0  bkt_pred=0.225  elo_pred=0.342
  skill=word_problem diff=6  correct=0  bkt_pred=0.226  elo_pred=0.328
  skill=word_problem diff=6  correct=1  bkt_pred=0.226  elo_pred=0.315
  skill=subtraction  diff=2  correct=0  bkt_pred=0.306  elo_pred=0.640


## Multi-Learner AUC Evaluation

In [ ]:
N_LEARNERS = 20
results = evaluate(loader.items, n_learners=N_LEARNERS)
print("\n" + "="*45)
print("  KNOWLEDGE TRACING EVALUATION RESULTS")
print("="*45)
print(f"  Learners simulated : {results['n_learners']}")
print(f"  Total observations : {results['n_observations']}")
print(f"  BKT  AUC           : {results['BKT_AUC']:.4f}")
print(f"  Elo  AUC           : {results['Elo_AUC']:.4f}")
print("="*45)


  KNOWLEDGE TRACING EVALUATION RESULTS
  Learners simulated : 20
  Total observations : 800
  BKT  AUC           : 0.4978
  Elo  AUC           : 0.5272
  BKT wins           : False


## Per-Skill BKT Trajectory

In [7]:
from tutor.adaptive import BKTTracker, AdaptiveSelector
import random

bkt = BKTTracker()
elo = EloTracker()
selector = AdaptiveSelector(loader.items)
rng = random.Random(42)
true_mastery = {s: rng.uniform(0.1, 0.5) for s in ["counting","number_sense","addition","subtraction","word_problem"]}

trajectory = {s: [] for s in true_mastery}
for step in range(50):
    item = selector.select(bkt)
    if not item:
        break
    skill = item["skill"]
    p_correct = max(0.05, min(0.95, true_mastery[skill] + rng.gauss(0, 0.1)))
    correct = rng.random() < p_correct
    bkt.update(skill, correct)
    elo.update(skill, item["difficulty"], correct)
    trajectory[skill].append(bkt.p_know(skill))
    true_mastery[skill] = min(0.95, true_mastery[skill] + 0.015)

print("\nFinal BKT mastery estimates:")
for skill, vals in trajectory.items():
    if vals:
        bar = "█" * int(vals[-1] * 20)
        print(f"  {skill:14s}  {bar:20s}  {vals[-1]:.3f}")


Final BKT mastery estimates:
  counting        ████████████████      0.824
  number_sense    ██                    0.149
  addition        ████████████          0.649
  subtraction     ████████████████      0.840
  word_problem    █████████████         0.687


## Save Results

In [8]:
out = Path("reports")
out.mkdir(exist_ok=True)
with open(out / "kt_eval_results.json", "w") as f:
    json.dump(results, f, indent=2)
